# SFINCS factorial design – scenario generation and output extraction (San Felipe)

This notebook builds the same quadtree SFINCS model as `Build_model.ipynb` and then generates the **2³ = 8 scenario runs** of the factorial design. The three drivers are synthetic and each is set to a lower or an upper bound:

- **P**: precipitation
- **S**: storm surge
- **I**: initial infiltration rate in the urban (built-up) area

Wind is not a driver: the domain is too small for local wind set-up, and large-scale wind effects are captured by the storm surge.

## Workflow

1. **Generate the runs** (sections 1–10). The model is built once and one complete SFINCS model per scenario is written to `sfincs_runs/run_<name>/`. **This creates 8 folders.**
2. **Run SFINCS** in every run folder. This is done outside this notebook.
3. **Extract the scores** (section 11). The history output (`sfincs_his.nc`) of every run is read and the flood scores Y1–Y4 are written to `Y1.csv` … `Y4.csv`. This only works after step 2.
4. The factorial analysis itself (effects of P, S and I on Y1–Y4) is done in a separate notebook that reads these CSV files.

## Notebook outline

| Section | Content |
|---|---|
| 1. Imports and paths | Input files (`data/`), global parameters, file check |
| 2. Initialise model | Model object, simulation period, initial water level |
| 3. Grid | Rotated quadtree grid, refined over the LiDAR extent |
| 4. Elevation | LiDAR with bathymetry as fallback |
| 5. Mask | Active cells and offshore water level boundary |
| 6. Structures | Buildings represented as raised grid cells |
| 7. Observation points | Points at which the flood scores are computed |
| 8. Manning roughness | From ESA WorldCover |
| 9. Drivers | Bounds, scenario table and plots of the lower/upper bound of P, S and I |
| 10. Scenario runs | File generator loop: writes the 8 run folders |
| 11. Flood scores | Y1–Y4 from the SFINCS output (after running SFINCS) |

Sections 1–8 are the same as in `Build_model.ipynb`. All model data are in **EPSG:32616** (UTM zone 16N) and all times are in **UTC**.

## 1. Imports and paths

All input files are read from the `data/` folder (the same folder as for `Build_model.ipynb`) and defined once in the path cell below.

| File in `data/` | Section | Description |
|---|---|---|
| `Lidar_gecorrigeerd_SanFelipe.tif` | 3, 4 | Corrected, gap-free LiDAR DEM (UTM 16N). First elevation source; its extent defines the refinement area |
| `gecombineerde_bathymetrie.tif` | 4 | Combined bathymetry/topography raster; fills everything the LiDAR does not cover |
| `SanFelipe_buildings.gpkg` | 6 | Building footprints |
| `street_points.csv` | 7 | Observation points (columns `Name`, `X` = longitude, `Y` = latitude, WGS84) |
| `ESA_WorldCover_10m_2021_v200_N21W090_Map.tif` | 8, 10 | ESA WorldCover 2021 land cover (10 m) |

The forcing is synthetic, so no meteorological or water level files are needed. `worldcover_manning.csv` is generated by section 8 and written to `data/`.

Output folders: the base model is written to `model_root` and the scenario runs to `runs_dir`.

In [ ]:
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import xarray as xr
from hydromt_sfincs import SfincsModel
from pyproj import Transformer
from shapely import affinity
from shapely.geometry import LineString, Point, box

In [ ]:
data_dir = Path("data")

lidar_tif = data_dir / "Lidar_gecorrigeerd_SanFelipe.tif"
topography_tif = data_dir / "gecombineerde_bathymetrie.tif"
buildings_gpkg = data_dir / "SanFelipe_buildings.gpkg"
street_points_csv = data_dir / "street_points.csv"
worldcover_tif = data_dir / "ESA_WorldCover_10m_2021_v200_N21W090_Map.tif"

manning_csv = data_dir / "worldcover_manning.csv"

model_root = Path("sfincs_san_felipe_qtr_stat_analysis")
runs_dir = Path("sfincs_runs")

### Parameters

The simulation lasts 3 days. Because the forcing is synthetic, the calendar dates are arbitrary; only the duration and the time of the peak (middle of the simulation) matter.

Mean sea level is not 0 in the elevation datum of the model but at `msl_offset` = −0.5 m. This value is used as initial water level and as offset on the boundary water level.

The bounds of the drivers are set in section 9.

In [ ]:
tstart, tstop = "20240204 000000", "20240207 000000"  # simulation period, UTC (YYYYMMDD HHMMSS)

target_crs = "EPSG:32616"  # model CRS (UTM zone 16N)
lat_min, lat_max = 21.562176, 21.607176  # domain extent, WGS84 [deg]
lon_min, lon_max = -88.251847, -88.181847
rotation = 35.0  # rotation of the domain around its centre [deg, counter-clockwise]
res = 40.0  # coarsest cell size [m]
refine_level = 3  # refinement levels inside the LiDAR extent, each level halves the cell size

msl_offset = -0.5  # mean sea level in the model elevation datum [m]
time_margin = pd.Timedelta(hours=1)  # extra time before and after the simulation for the water level series

required = [lidar_tif, topography_tif, buildings_gpkg, street_points_csv, worldcover_tif]
missing = [str(p) for p in required if not p.exists()]
assert not missing, f"Missing input files: {missing}"

model_root.mkdir(exist_ok=True)

## 2. Initialise model

Creates the SFINCS model object in write mode (`w+`) in `model_root`; an existing model in that folder is overwritten. `write_gis=True` also writes GIS files of the model layers.

The config sets the quadtree grid file, switches SnapWave off and defines the simulation period (`tref`, `tstart`, `tstop`). Advection and Coriolis are switched off. The precipitation time series file is registered as `sfincs.precip`.

In [ ]:
sf = SfincsModel(root=model_root, mode="w+", write_gis=True)
sf.config.update({"snapwave": 0, "qtrfile": "sfincs.qtr"})
sf.config.update_grid_from_config()

sf.config.update({
    "tref": tstart,
    "tstart": tstart,
    "tstop": tstop,
    "zsini": msl_offset,
    "advection": 0,  # 0 = off
    "coriolis": 0,  # 0 = off
    "precipfile": "sfincs.precip",
})

## 3. Grid

The model domain is defined as a lat/lon box (`lat_min` … `lon_max`), converted to UTM and rotated by `rotation` degrees around its centre.

The quadtree grid starts with cells of `res` metres and is refined inside the extent of the LiDAR raster. Each refinement level halves the cell size (`refine_level = 3`: 40 m → 5 m). The printed cell sizes confirm the result. The LiDAR raster must have a CRS equal to `target_crs`.

In [ ]:
t = Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)
x_min, y_min = t.transform(lon_min, lat_min)
x_max, y_max = t.transform(lon_max, lat_max)
x_c, y_c = (x_min + x_max) / 2, (y_min + y_max) / 2

domain_rot = affinity.rotate(box(x_min, y_min, x_max, y_max), rotation, origin=(x_c, y_c))
region_gdf = gpd.GeoDataFrame(geometry=[domain_rot], crs=target_crs)

with rasterio.open(lidar_tif) as src:
    assert src.crs is not None, "LiDAR raster has no CRS"
    lidar_bounds = src.bounds

refinement_polygons = gpd.GeoDataFrame(
    {"refinement_level": [refine_level]},
    geometry=[box(*lidar_bounds)],
    crs=target_crs,
)

sf.quadtree_grid.create_from_region(
    region={"geom": region_gdf},
    res=res,
    crs=target_crs,
    rotated=True,
    refinement_polygons=refinement_polygons,
)

cell_size = np.sqrt(np.asarray(sf.quadtree_grid.data.ugrid.grid.area))
print(f"Cells: {cell_size.size} | cell size min / max: {cell_size.min():.1f} / {cell_size.max():.1f} m")

## 4. Elevation

Elevation is interpolated onto the quadtree cells from a prioritised list of rasters: the first raster has priority and the following ones only fill cells without data. The LiDAR comes first and the coarser bathymetry fills everything outside the LiDAR coverage. `buffer_cells=1` reads the rasters slightly beyond the domain to avoid edge effects.

The printed check shows the elevation range and the number of cells without data (NaN).

In [ ]:
elevation_list = [
    {"elevation": os.path.abspath(lidar_tif)},
    {"elevation": os.path.abspath(topography_tif)},
]

sf.quadtree_elevation.create(elevation_list=elevation_list, buffer_cells=1)
sf.quadtree_elevation.write()

z = sf.quadtree_elevation.data["z"]
print(f"z min / mean / max: {float(z.min()):.2f} / {float(z.mean()):.2f} / {float(z.max()):.2f} m")
print(f"Cells without data (NaN): {int(np.isnan(z.values).sum())} of {z.size}")

## 5. Mask

The mask defines the role of every cell: `0` inactive, `1` active, `2` water level boundary.

- **Active cells:** all cells inside the domain with an elevation above `zmin`. With `zmin = -50` m this is the whole domain (the deepest point is about −12.7 m).
- **Boundary cells:** cells below `boundary_zmax` inside a strip of `strip_width` metres along the offshore edge of the domain (north-west side after rotation). The strip is built on the unrotated domain and then rotated with the same angle and origin as the domain.

In [ ]:
zmin = -50.0  # cells below this elevation are inactive [m]
boundary_zmax = 0.0  # only cells below this elevation can become boundary cells [m]
strip_width = 100.0  # width of the boundary strip along the offshore edge [m]

sf.quadtree_mask.create_active(zmin=zmin, include_polygon=region_gdf, reset_mask=True)
mask = sf.quadtree_grid.data["mask"]
n_active = int((mask.values == 1).sum())
assert n_active > 0, "No active cells"
print(f"Active cells: {n_active} of {mask.size}")

strip = box(x_min, y_max - strip_width, x_max, y_max + 2 * res)
strip_gdf = gpd.GeoDataFrame(geometry=[affinity.rotate(strip, rotation, origin=(x_c, y_c))], crs=target_crs)

sf.quadtree_mask.create_boundary(
    btype="waterlevel",
    zmax=boundary_zmax,
    include_polygon=strip_gdf,
    reset_bounds=True,
)
sf.quadtree_mask.write()

### Quick visualisation of the domain

Satellite basemap with the model region (needs an internet connection), followed by the elevation on the quadtree grid.

In [ ]:
sf.plot_basemap(
    variable="mask",
    shaded=True,
    plot_region=True,
    plot_bounds=False,
    bmap="sat",
    zoomlevel=13,
    figsize=(12, 8),
    alpha=0.0,
)
plt.show()

fig, ax = plt.subplots(figsize=(12, 8))
sf.quadtree_elevation.data["z"].ugrid.plot(ax=ax, cmap="terrain", vmin=-2, vmax=5, edgecolors="none")
ax.set_aspect("equal")
ax.set_title("Elevation and quadtree refinement")
plt.show()

## 6. Structures

Buildings are added as raised cells: every grid cell whose centre lies inside a selected building is raised by `building_raise` metres, so that the buildings act as obstacles to the flow.

Only buildings above a slanted cut-off line are used. The line is defined relative to the centre of the building footprints:

- `y_offset`: shifts the line up (+) or down (−) [m]
- `line_angle`: angle of the line [degrees]

The elevation is re-created from the rasters before the cells are raised, so re-running this cell never adds the increase twice. The plot shows the result together with the cut-off line.

In [ ]:
y_offset = -190  # shift of the cut-off line relative to the centre of the buildings [m]
line_angle = 167.0  # angle of the cut-off line [deg]
building_raise = 3.5  # elevation increase of cells under selected buildings [m]

buildings_all = gpd.read_file(buildings_gpkg).to_crs(target_crs)

minx_b, miny_b, maxx_b, maxy_b = buildings_all.total_bounds
center_x, center_y = (minx_b + maxx_b) / 2, (miny_b + maxy_b) / 2
target_y = center_y + y_offset

unrotated = buildings_all.geometry.apply(
    lambda geom: affinity.rotate(geom, -line_angle, origin=(center_x, center_y))
)
buildings_gdf = buildings_all[unrotated.centroid.y > target_y].copy()

sf.quadtree_elevation.create(elevation_list=elevation_list, buffer_cells=1)
z = sf.quadtree_elevation.data["z"]

x_faces, y_faces = z.ugrid.grid.face_x, z.ugrid.grid.face_y
grid_gdf = gpd.GeoDataFrame(
    {"cell_index": np.arange(len(x_faces))},
    geometry=gpd.points_from_xy(x_faces, y_faces),
    crs=target_crs,
)
joined = gpd.sjoin(grid_gdf, buildings_gdf, how="inner", predicate="intersects")
building_cells = joined["cell_index"].unique()

z.values[building_cells] += building_raise
sf.quadtree_elevation.data["z"] = z
print(f"Buildings selected: {len(buildings_gdf)} of {len(buildings_all)} | cells raised: {len(building_cells)}")

fig, ax = plt.subplots(figsize=(10, 10))
z.ugrid.plot(ax=ax, cmap="terrain", vmin=-3, vmax=9, edgecolors="none")
cutoff = affinity.rotate(
    LineString([(minx_b - 500, target_y), (maxx_b + 500, target_y)]),
    line_angle,
    origin=(center_x, center_y),
)
ax.plot(*cutoff.xy, "r--", lw=2, label="Cut-off line")
ax.set_xlim(minx_b - 400, maxx_b + 400)
ax.set_ylim(miny_b - 400, maxy_b + 400)
ax.set_aspect("equal")
ax.set_title("San Felipe: cut-off line and raised buildings")
ax.legend(loc="upper right")
plt.show()

## 7. Observation points

Output points at which SFINCS writes time series (`sfincs_his.nc`) when the model is run. The points are the manual points defined in the cell below plus all points in `street_points.csv`. All coordinates are lon/lat (WGS84) and are converted to the model CRS.

In [ ]:
manual_points = {  # name: (longitude, latitude)
    "Main street": (-88.232373, 21.565493),
    "Station_SF": (-88.215227, 21.572000),
}

obs_manual = gpd.GeoDataFrame(
    {"name": list(manual_points)},
    geometry=[Point(xy) for xy in manual_points.values()],
    crs="EPSG:4326",
)

df_csv = pd.read_csv(street_points_csv)
obs_csv = gpd.GeoDataFrame(
    {"name": df_csv["Name"]},
    geometry=gpd.points_from_xy(df_csv["X"], df_csv["Y"]),
    crs="EPSG:4326",
)

obs_gdf = pd.concat([obs_manual, obs_csv], ignore_index=True).to_crs(target_crs)
sf.observation_points.set(obs_gdf)
print(f"Observation points: {len(obs_gdf)}")

fig, ax = plt.subplots(figsize=(10, 8))
z.ugrid.plot(ax=ax, cmap="terrain", vmin=-3, vmax=5, edgecolors="none")
obs_gdf.plot(ax=ax, color="red", marker="*", markersize=150, edgecolor="white", label="Observation points")
ax.set_aspect("equal")
ax.set_title("Elevation and observation points")
ax.legend(loc="upper right")
plt.show()

## 8. Manning roughness

Manning's *n* is assigned from the ESA WorldCover land cover map with a reclass table (`lulc` class → `N`). The table is defined in the cell below, written to `worldcover_manning.csv` and read by HydroMT-SFINCS. Roughness is the same in all scenarios. Infiltration is not set here, because it is a driver and differs per scenario (section 9.4).

| WorldCover class | N [s/m^(1/3)] |
|---|---|
| 10 Tree cover | 0.12 |
| 20 Shrubland | 0.05 |
| 30 Grassland | 0.034 |
| 40 Cropland | 0.037 |
| 50 Built-up | 0.1 |
| 60 Bare / sparse vegetation | 0.023 |
| 80 Permanent water | 0.02 |
| 90 Herbaceous wetland | 0.035 |
| 95 Mangroves | 0.07 |
| 100 Moss and lichen | 0.025 |

In [ ]:
manning = pd.DataFrame({
    "lulc": [10, 20, 30, 40, 50, 60, 80, 90, 95, 100],
    "N": [0.12, 0.05, 0.034, 0.037, 0.1, 0.023, 0.02, 0.035, 0.07, 0.025],
}).set_index("lulc")
manning.to_csv(manning_csv)

roughness_list = [{
    "lulc": os.path.abspath(worldcover_tif),
    "reclass_table": os.path.abspath(manning_csv),
}]
sf.quadtree_roughness.create(roughness_list=roughness_list)
sf.quadtree_roughness.write()

fig, ax = plt.subplots(figsize=(10, 8))
sf.quadtree_roughness.data["manning"].ugrid.plot(
    ax=ax, cmap="tab10", edgecolors="none", cbar_kwargs={"label": "Manning n [s/m^(1/3)]"}
)
ax.set_aspect("equal")
ax.set_title("Manning roughness")
plt.show()

## 9. Drivers

The three drivers each have a lower (−) and an upper (+) bound. A full 2³ factorial design combines them in 8 runs, named after the drivers that are at their upper bound (`(1)` = all at the lower bound).

### 9.1 Bounds and scenarios

| Driver | Lower bound (−) | Upper bound (+) | Description |
|---|---|---|---|
| P precipitation | 0.0 mm/h | 13.4 mm/h | Peak intensity of a Gaussian event. The upper bound gives a total of about 306 mm in 3 days (50-year return period of the 3-day cumulative precipitation) |
| S storm surge | 0.0 m | 0.87 m | Peak of a Gaussian surge on top of a synthetic tide (50-year return period). The lower bound is the tide only |
| I infiltration | 0.0 mm/h | 50 mm/h | Infiltration rate of the built-up class (WorldCover 50). 0 is fully paved; 50 mm/h equals the grassland class |

| Run | P | S | I |
|---|---|---|---|
| (1) | − | − | − |
| p | + | − | − |
| s | − | + | − |
| i | − | − | + |
| ps | + | + | − |
| pi | + | − | + |
| si | − | + | + |
| psi | + | + | + |

Gaussian events are used to avoid sudden shocks in the model. `sigma_surge` and `sigma_precip` are the standard deviations (widths) of the surge and precipitation curves. All drivers peak at the middle of the simulation.

In [ ]:
BOUNDS = {
    "P": {"low": 0.0, "high": 13.4},  # peak precipitation intensity [mm/h]
    "S": {"low": 0.0, "high": 0.87},  # peak storm surge [m]
    "I": {"low": 0.0, "high": 50.0},  # infiltration rate of the built-up class [mm/h]
}
sigma_surge = 6.0  # standard deviation of the Gaussian storm surge [h]
sigma_precip = 9.1  # standard deviation of the Gaussian precipitation event [h]
m2_amplitude, m2_period = 0.16, 12.42  # synthetic tide, M2 constituent [m], [h]
k1_amplitude, k1_period = 0.14, 23.93  # synthetic tide, K1 constituent [m], [h]

FACTORIAL_RUNS = {  # 0 = lower bound (-), 1 = upper bound (+)
    "(1)": {"P": 0, "S": 0, "I": 0},
    "p": {"P": 1, "S": 0, "I": 0},
    "s": {"P": 0, "S": 1, "I": 0},
    "i": {"P": 0, "S": 0, "I": 1},
    "ps": {"P": 1, "S": 1, "I": 0},
    "pi": {"P": 1, "S": 0, "I": 1},
    "si": {"P": 0, "S": 1, "I": 1},
    "psi": {"P": 1, "S": 1, "I": 1},
}


def driver_values(levels):
    return tuple(BOUNDS[d]["high"] if levels[d] else BOUNDS[d]["low"] for d in ("P", "S", "I"))


scenarios = pd.DataFrame.from_dict(
    {run: driver_values(levels) for run, levels in FACTORIAL_RUNS.items()},
    orient="index",
    columns=["P peak [mm/h]", "S peak [m]", "I built-up [mm/h]"],
)
print(scenarios)

Time axes shared by all drivers: the water level is defined every 10 minutes from `time_margin` before the start to `time_margin` after the end of the simulation, precipitation every hour over the simulation. `t_peak` is the middle of the simulation.

In [ ]:
t0_mod, t1_mod = (pd.Timestamp(t) for t in sf.get_model_time())
tref = pd.Timestamp(tstart)
t_peak = t0_mod + (t1_mod - t0_mod) / 2
t_peak_sec = (t_peak - tref).total_seconds()

time_wl = pd.date_range(t0_mod - time_margin, t1_mod + time_margin, freq="10min")
time_p = pd.date_range(t0_mod, t1_mod, freq="1h")
time_sec_p = (time_p - tref).total_seconds().values
sim_hours = (t1_mod - t0_mod).total_seconds() / 3600.0

### 9.2 Storm surge (S) and tide

The water level at the offshore boundary is a synthetic micro-tidal tide (M2 + K1, high water of +0.30 m at `t_peak`) plus a Gaussian storm surge. At the lower bound the surge is 0, so only the tide remains. The series is relative to mean sea level; `msl_offset` shifts it to the model datum when it is assigned to the model (section 10). The plot shows the lower and upper bound.

In [ ]:
def water_level(surge_peak):
    dt_sec = np.asarray((time_wl - t_peak).total_seconds())
    m2 = m2_amplitude * np.cos(2 * np.pi * dt_sec / (m2_period * 3600))
    k1 = k1_amplitude * np.cos(2 * np.pi * dt_sec / (k1_period * 3600))
    dt_h = dt_sec / 3600.0
    surge = surge_peak * np.exp(-0.5 * (dt_h / sigma_surge) ** 2)
    return m2 + k1, surge


hours_wl = np.asarray((time_wl - tref).total_seconds()) / 3600.0
tide, _ = water_level(0.0)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(hours_wl, tide, "k--", alpha=0.6, label="Astronomical tide")
for name, color in (("low", "tab:blue"), ("high", "tab:red")):
    surge_peak = BOUNDS["S"][name]
    tide_, surge = water_level(surge_peak)
    label = f"{'Lower' if name == 'low' else 'Upper'} bound: tide + surge (peak {surge_peak:.2f} m)"
    ax.plot(hours_wl, tide_ + surge, color=color, lw=2, label=label)
ax.set_xlim(0, sim_hours)
ax.set_xlabel("Simulation time [h]")
ax.set_ylabel("Water level [m MSL]")
ax.set_title("Offshore water level boundary (S)")
ax.grid(alpha=0.3)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 9.3 Precipitation (P)

Spatially uniform precipitation with a Gaussian intensity curve (hourly values, in mm/h). The upper bound is chosen such that the total over the 3 days is about 306 mm. The lower bound is no rain. The series is written to `sfincs.precip` in every run folder (section 10).

In [ ]:
def precipitation(p_peak):
    dt_h = (time_sec_p - t_peak_sec) / 3600.0
    return p_peak * np.exp(-0.5 * (dt_h / sigma_precip) ** 2)


hours_p = time_sec_p / 3600.0
p_low, p_high = precipitation(BOUNDS["P"]["low"]), precipitation(BOUNDS["P"]["high"])

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(hours_p, p_high, width=0.8, color="tab:red", alpha=0.6,
       label=f"Upper bound: peak {BOUNDS['P']['high']:.1f} mm/h, total {p_high.sum():.0f} mm")
ax.plot(hours_p, p_low, color="tab:blue", lw=2,
        label=f"Lower bound: peak {BOUNDS['P']['low']:.1f} mm/h, total {p_low.sum():.0f} mm")
ax.set_xlim(0, sim_hours)
ax.set_xlabel("Simulation time [h]")
ax.set_ylabel("Precipitation intensity [mm/h]")
ax.set_title("Uniform precipitation (P)")
ax.grid(alpha=0.3)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 9.4 Infiltration (I)

Infiltration is a spatially varying, constant-in-time rate `qinf` [mm/h] per WorldCover class (reclass table `lulc` → `qinf`). Only the built-up class (50) is a driver; all other classes are the same in every run.

| WorldCover class | qinf [mm/h] |
|---|---|
| 10 Tree cover | 50 |
| 20 Shrubland | 100 |
| 30 Grassland | 50 |
| 40 Cropland | 20 |
| 50 Built-up | **driver I** (0 or 50) |
| 60 Bare / sparse vegetation | 1994.4 |
| 80 Permanent water | 0 |
| 90 Herbaceous wetland | 0.072 |
| 95 Mangroves | 0.072 |
| 100 Moss and lichen | 0 |

In [ ]:
def infiltration_table(qinf_city):
    return pd.DataFrame({
        "lulc": [10, 20, 30, 40, 50, 60, 80, 90, 95, 100],
        "qinf": [50.0, 100.0, 50.0, 20.0, qinf_city, 1994.4, 0.0, 0.072, 0.072, 0.0],
    }).set_index("lulc")


values = [BOUNDS["I"]["low"], BOUNDS["I"]["high"]]

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Lower bound", "Upper bound"], values, color=["tab:blue", "tab:red"], alpha=0.8)
for k, v in enumerate(values):
    ax.text(k, v, f"{v:.1f} mm/h", ha="center", va="bottom")
ax.set_ylim(0, max(values) * 1.15)
ax.set_ylabel("Infiltration rate [mm/h]")
ax.set_title("Infiltration of the built-up class (I)")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 10. Scenario runs (file generator loop)

> **This cell creates 8 folders.** For every scenario it writes a complete SFINCS model (grid, elevation, mask, roughness, infiltration, observation points, forcing, `sfincs.inp`) to `runs_dir/run_<name>/`, so the disk space needed is about 8 times that of one model. Existing run folders are overwritten. SFINCS is **not** run here.

The model built in sections 1–8 stays in memory and is reused for all scenarios. For each run the loop:

1. points the model to a new folder (`sf.root`);
2. sets the water level boundary (tide + surge, shifted by `msl_offset`);
3. writes the precipitation time series to `sfincs.precip`;
4. writes the infiltration reclass table (`worldcover_infiltration.csv`) and sets the infiltration;
5. updates the config and writes the model;
6. edits `sfincs.inp`: references to NetCDF meteo forcing (`netamuamvfile`, `netamprfile`) are removed, so SFINCS only uses `sfincs.precip`, and `prcpfile` is added if it is missing.

`ModelRootWrapper` is a small helper class that makes HydroMT-SFINCS (v1) accept a different root folder for each scenario. The precipitation file is registered as `precipfile` (section 2) and as `prcpfile` (step 5 and 6); both point to `sfincs.precip`. `dtmapout` is the interval of the map output (900 s) and `storecumprcp = 1` stores the cumulative precipitation in the map output.

After this section, run SFINCS in every `run_<name>` folder.

In [ ]:
class ModelRootWrapper:
    def __init__(self, path):
        self.path = Path(path).resolve()
        self.mode = "w+"

    def is_reading_mode(self):
        return False

    def is_writing_mode(self):
        return True

    def _assert_write_mode(self):
        pass

    def __str__(self):
        return str(self.path)

    def __fspath__(self):
        return str(self.path)

    def __truediv__(self, other):
        return self.path / other


minx, miny, maxx, maxy = region_gdf.to_crs(4326).total_bounds
locations = gpd.GeoDataFrame({"index": [1]}, geometry=[Point((minx + maxx) / 2, maxy)], crs=4326)

runs_dir.mkdir(parents=True, exist_ok=True)

for run_name, levels in FACTORIAL_RUNS.items():
    print(f"Writing run_{run_name} ...")
    p_peak, surge_peak, qinf_city = driver_values(levels)

    run_dir = runs_dir / f"run_{run_name}"
    run_dir.mkdir(parents=True, exist_ok=True)
    sf.root = ModelRootWrapper(run_dir)

    tide, surge = water_level(surge_peak)
    sf.water_level.create(
        timeseries=pd.DataFrame({1: tide + surge}, index=time_wl),
        locations=locations,
        offset=msl_offset,
        buffer=100000,
        merge=False,
    )

    precip_data = np.column_stack((time_sec_p, precipitation(p_peak)))
    np.savetxt(run_dir / "sfincs.precip", precip_data, fmt="%.1f   %.6f")

    infiltration_csv = run_dir / "worldcover_infiltration.csv"
    infiltration_table(qinf_city).to_csv(infiltration_csv)
    sf.quadtree_infiltration.create_constant(
        lulc=os.path.abspath(worldcover_tif),
        reclass_table=os.path.abspath(infiltration_csv),
        reproj_method="average",
    )

    sf.config.update({"dtmapout": 900, "storecumprcp": 1, "prcpfile": "sfincs.precip"})
    sf.write()

    inp_path = run_dir / "sfincs.inp"
    lines = [
        line for line in inp_path.read_text().splitlines()
        if not line.strip().startswith(("netamuamvfile", "netamprfile"))
    ]
    inp_text = "\n".join(lines)
    if "prcpfile" not in inp_text:
        inp_text += "\nprcpfile = sfincs.precip"
    inp_path.write_text(inp_text + "\n")

print(f"\n{len(FACTORIAL_RUNS)} runs written to {runs_dir.resolve()}")

## 11. Flood scores (Y1–Y4)

> **Run this section only after SFINCS has been run in all 8 run folders.** It reads `sfincs_his.nc` from every `run_<name>` folder, so it fails or skips runs when that file does not exist yet.

For every run, four indicators of the flood extent are computed from the water depth (`point_h`) at the observation points. A point counts as *flooded* when its maximum depth exceeds `h_thresh`. Points in `exclude_stations` are left out (`Station_SF` lies in the mangroves, not in the town).

| Score | Definition | Unit |
|---|---|---|
| Y1 | Percentage of observation points that are flooded at some time during the run | % |
| Y2 | Mean of the maximum depth over the flooded points | m |
| Y3 | Mean total inundation duration over the flooded points (number of output steps above `h_thresh` times the output interval) | h |
| Y4 | 95th percentile of the maximum depth over the flooded points | m |

The scores are written to `Y1.csv`, `Y2.csv`, `Y3.csv` and `Y4.csv` (columns `run` and the score) in `results_dir`. These files are the input of the factorial analysis in the separate notebook. All 8 runs are needed for that analysis.

In [ ]:
h_thresh = 0.01  # water depth above which an observation point counts as flooded [m]
exclude_stations = ["Station_SF"]  # observation points left out of the scores
results_dir = Path(".")  # folder where Y1.csv ... Y4.csv are written


def flood_scores(his_path):
    with xr.open_dataset(his_path) as ds:
        names = np.array([
            n.decode("utf-8").strip() if isinstance(n, bytes) else str(n).strip()
            for n in ds["station_name"].values
        ])
        valid = ~np.isin(names, exclude_stations)
        point_h = ds["point_h"].values[:, valid]
        time_vals = ds["time"].values

    dt_hours = float(np.mean(np.diff(time_vals)) / np.timedelta64(1, "h")) if len(time_vals) > 1 else 10.0 / 60.0

    n_obs = int(valid.sum())
    hmax = np.nanmax(point_h, axis=0)
    duration = np.sum(point_h > h_thresh, axis=0) * dt_hours
    flooded = hmax > h_thresh
    n_flooded = int(flooded.sum())

    return {
        "total_obs": n_obs,
        "n_flooded_obs": n_flooded,
        "Y1": n_flooded / n_obs * 100.0 if n_obs > 0 else 0.0,
        "Y2": float(np.mean(hmax[flooded])) if n_flooded > 0 else 0.0,
        "Y3": float(np.mean(duration[flooded])) if n_flooded > 0 else 0.0,
        "Y4": float(np.percentile(hmax[flooded], 95)) if n_flooded > 0 else 0.0,
    }


rows = []
for run_name, levels in FACTORIAL_RUNS.items():
    his_path = runs_dir / f"run_{run_name}" / "sfincs_his.nc"
    if not his_path.exists():
        print(f"run_{run_name:>4}: sfincs_his.nc not found (has SFINCS been run?)")
        continue

    scores = flood_scores(his_path)
    rows.append({"run": run_name, **{d: "+" if levels[d] else "-" for d in levels}, **scores})
    print(
        f"run_{run_name:>4}: Y1 = {scores['Y1']:5.1f} % | Y2 = {scores['Y2']:.3f} m | "
        f"Y3 = {scores['Y3']:5.1f} h | Y4 = {scores['Y4']:.3f} m"
    )

df_summary = pd.DataFrame(rows)

if df_summary.empty:
    print("\nNo output found, nothing written.")
else:
    for y in ("Y1", "Y2", "Y3", "Y4"):
        df_summary[["run", y]].to_csv(results_dir / f"{y}.csv", index=False)
    print(f"\nWritten: Y1.csv, Y2.csv, Y3.csv, Y4.csv to {results_dir.resolve()}")
    if len(df_summary) < len(FACTORIAL_RUNS):
        print(f"Warning: only {len(df_summary)} of {len(FACTORIAL_RUNS)} runs found; the factorial analysis needs all runs.")
    print(df_summary[["run", "P", "S", "I", "Y1", "Y2", "Y3", "Y4"]].to_string(index=False))